In [1]:
# imports
import csv
import sqlite3
import time
import hashlib
import pickle
import json
import logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List, Tuple, Set, Any
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Paths
ROOT = Path('.')
GO_DIR = ROOT / 'GreatOutdoors'
TARGET_DB_PATH = ROOT / 'GreatOutdoors_SDM.db'

# Source configuration
SQLITE_SOURCES = [
    ('CRM-data.sqlite', 'CRM_'),
    ('GO_SALES-data.sqlite', 'SALES_'),
    ('GO_STAFF-data.sqlite', 'STAFF_'),
]
CSV_SOURCES = [
    ('INVENTORY_LEVELS-data.csv', 'INVENTORY_LEVELS'),
    ('PRODUCT_FORECAST-data.csv', 'PRODUCT_FORECAST'),
    ('SALES_TARGET-data.csv', 'SALES_TARGET'),
]

# Composite keys for CSV targets
CSV_PRIMARY_KEYS = {
    'INVENTORY_LEVELS': ['INVENTORY_YEAR', 'INVENTORY_MONTH', 'PRODUCT_NUMBER'],
    'PRODUCT_FORECAST': ['PRODUCT_NUMBER', 'YEAR', 'MONTH'],
    'SALES_TARGET': ['SALES_STAFF_CODE', 'SALES_YEAR', 'SALES_PERIOD', 'RETAILER_CODE', 'PRODUCT_NUMBER'],
}

In [2]:
# Helper Functions: Schema and Hashing

def compute_row_hash(row_dict: Dict[str, Any]) -> str:
    """Compute MD5 hash of row data for change detection."""
    row_str = json.dumps(row_dict, sort_keys=True, default=str)
    return hashlib.md5(row_str.encode()).hexdigest()


def detect_csv_encoding(csv_path: Path) -> str:
    """Detect CSV file encoding."""
    encodings = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']
    
    for encoding in encodings:
        try:
            with open(csv_path, 'r', encoding=encoding) as f:
                f.read(1000)  # Try to read first 1000 chars
            return encoding
        except (UnicodeDecodeError, LookupError):
            continue
    
    logger.warning(f"Could not detect encoding for {csv_path}, using latin-1")
    return 'latin-1'


def get_sqlite_schema(db_path: Path) -> Dict[str, Dict[str, Any]]:
    """Extract table schemas and foreign keys from SQLite database."""
    schema = {}
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    try:
        # Get all tables
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables = cursor.fetchall()
        
        for (table_name,) in tables:
            if table_name.startswith('sqlite_'):
                continue
                
            schema[table_name] = {
                'columns': [],
                'primary_key': None,
                'foreign_keys': [],
                'indexes': []
            }
            
            # Get column info
            cursor.execute(f"PRAGMA table_info({table_name})")
            for cid, name, type_, notnull, dflt_value, pk in cursor.fetchall():
                schema[table_name]['columns'].append({
                    'name': name,
                    'type': type_,
                    'not_null': bool(notnull),
                    'default': dflt_value,
                    'pk': pk
                })
                if pk:
                    if schema[table_name]['primary_key'] is None:
                        schema[table_name]['primary_key'] = []
                    schema[table_name]['primary_key'].append((name, pk))
            
            if schema[table_name]['primary_key']:
                schema[table_name]['primary_key'].sort(key=lambda x: x[1])
                schema[table_name]['primary_key'] = [col[0] for col in schema[table_name]['primary_key']]
            
            # Get foreign keys
            cursor.execute(f"PRAGMA foreign_key_list({table_name})")
            for id_, seq, table, from_col, to_col, on_delete, on_update, match in cursor.fetchall():
                schema[table_name]['foreign_keys'].append({
                    'from_column': from_col,
                    'to_table': table,
                    'to_column': to_col,
                    'on_delete': on_delete,
                    'on_update': on_update
                })
            
            # Get indexes
            cursor.execute(f"PRAGMA index_list({table_name})")
            for seq, name, unique, origin, partial in cursor.fetchall():
                if origin == 'c':  # skip auto-created indexes
                    continue
                schema[table_name]['indexes'].append({
                    'name': name,
                    'unique': bool(unique),
                    'partial': bool(partial)
                })
    finally:
        conn.close()
    
    return schema


def infer_csv_types(csv_path: Path) -> Dict[str, str]:
    """Infer data types from CSV file (sample first 100 rows)."""
    types = {}
    encoding = detect_csv_encoding(csv_path)
    
    with open(csv_path, 'r', encoding=encoding) as f:
        reader = csv.DictReader(f)
        sample_rows = []
        for i, row in enumerate(reader):
            if i >= 100:
                break
            sample_rows.append(row)
    
    if not sample_rows:
        return types
    
    # Infer types for each column
    for col_name in sample_rows[0].keys():
        types[col_name] = 'TEXT'  # Default
        
        # Check if all values can be parsed as INTEGER
        try:
            for row in sample_rows:
                if row[col_name].strip():
                    int(row[col_name])
            types[col_name] = 'INTEGER'
            continue
        except (ValueError, TypeError):
            pass
        
        # Check if all values can be parsed as REAL
        try:
            for row in sample_rows:
                if row[col_name].strip():
                    float(row[col_name])
            types[col_name] = 'REAL'
            continue
        except (ValueError, TypeError):
            pass
    
    return types


def extract_primary_key_tuple(row_dict: Dict[str, Any], pk_columns: List[str]) -> Tuple:
    """Extract primary key tuple from row."""
    return tuple(row_dict.get(col) for col in pk_columns)


In [3]:
def save_row_hashes(conn: sqlite3.Connection, source_name: str, row_hashes: Dict[Tuple, str], source_type: str = 'unknown'):
    """Save row hashes to metadata."""
    cursor = conn.cursor()
    hashes_blob = pickle.dumps(row_hashes)
    
    cursor.execute("""
        INSERT OR REPLACE INTO sdm_load_metadata 
        (source_name, source_type, row_count, row_hashes, last_sync_time, updated_at)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (source_name, source_type, len(row_hashes), hashes_blob, datetime.now(), datetime.now()))
    
    conn.commit()


def get_row_hashes(conn: sqlite3.Connection, source_name: str) -> Dict[Tuple, str]:
    """Retrieve row hashes from metadata."""
    cursor = conn.cursor()
    
    cursor.execute("SELECT row_hashes FROM sdm_load_metadata WHERE source_name = ?", (source_name,))
    row = cursor.fetchone()
    
    if row and row[0]:
        return pickle.loads(row[0])
    return {}


def init_sdm_metadata(conn: sqlite3.Connection):
    """Initialize SDM metadata table."""
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS sdm_load_metadata (
            source_name TEXT PRIMARY KEY,
            source_type TEXT,
            row_count INTEGER,
            row_hashes BLOB,
            last_sync_time TIMESTAMP,
            updated_at TIMESTAMP
        )
    """)
    conn.commit()


def record_source_metadata(conn: sqlite3.Connection, source_name: str, source_type: str):
    """Record source metadata."""
    cursor = conn.cursor()
    
    # Initialize metadata table if not exists
    init_sdm_metadata(conn)
    
    cursor.execute("""
        INSERT OR IGNORE INTO sdm_load_metadata 
        (source_name, source_type, row_count, last_sync_time, updated_at)
        VALUES (?, ?, ?, ?, ?)
    """, (source_name, source_type, 0, datetime.now(), datetime.now()))
    
    conn.commit()


def initialize_sdm_database(force_recreate: bool = False) -> sqlite3.Connection:
    """Initialize SDM database."""
    
    if force_recreate and TARGET_DB_PATH.exists():
        TARGET_DB_PATH.unlink()
        logger.info(f"Recreating SDM database: {TARGET_DB_PATH}")
    
    conn = sqlite3.connect(TARGET_DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON")
    
    # Initialize metadata table
    init_sdm_metadata(conn)
    
    return conn


def is_initial_load(conn: sqlite3.Connection) -> bool:
    """Check if this is the first load."""
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM sdm_load_metadata")
    count = cursor.fetchone()[0]
    return count == 0


def verify_sdm_schema(conn: sqlite3.Connection) -> Dict[str, Any]:
    """Verify SDM schema against source schemas."""
    results = {
        'schema_matches': {},
        'total_tables': 0,
        'matches': 0
    }
    
    cursor = conn.cursor()
    
    for source_filename, table_prefix in SQLITE_SOURCES:
        source_path = GO_DIR / source_filename
        if not source_path.exists():
            continue
        
        source_schema = get_sqlite_schema(source_path)
        
        for source_table, schema_info in source_schema.items():
            target_table = f"{table_prefix}{source_table}"
            results['total_tables'] += 1
            
            try:
                cursor.execute(f"PRAGMA table_info({target_table})")
                sdm_cols = cursor.fetchall()
                source_cols = schema_info['columns']
                
                if len(sdm_cols) == len(source_cols):
                    results['schema_matches'][target_table] = True
                    results['matches'] += 1
                else:
                    results['schema_matches'][target_table] = False
            except Exception as e:
                results['schema_matches'][target_table] = False
    
    return results


def generate_sdm_report(load_result: Dict[str, Any]) -> str:
    """Generate formatted load report."""
    report = f"""
╔════════════════════════════════════════════════════════════════╗
║              SDM LOAD REPORT                                   ║
╚════════════════════════════════════════════════════════════════╝

Load Type:        {load_result['load_type'].upper()}
Status:           {load_result['status'].upper()}
Elapsed Time:     {load_result['elapsed_seconds']:.2f} seconds
Tables Processed: {load_result['tables_loaded']}

SUMMARY BY TABLE:
{'-' * 70}
"""
    
    total_inserted = 0
    total_updated = 0
    total_deleted = 0
    
    for table_name, (inserted, updated, deleted) in sorted(load_result['table_results'].items()):
        total_inserted += inserted
        total_updated += updated
        total_deleted += deleted
        report += f"{table_name:45} +{inserted:6} ~{updated:6} -{deleted:6}\n"
    
    report += f"{'-' * 70}\n"
    report += f"{'TOTAL':45} +{total_inserted:6} ~{total_updated:6} -{total_deleted:6}\n\n"
    
    return report


def display_sdm_info(conn: sqlite3.Connection):
    """Display SDM database information."""
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT name FROM sqlite_master 
        WHERE type='table' AND name NOT LIKE 'sqlite_%' AND name != 'sdm_load_metadata'
    """)
    tables = cursor.fetchall()
    
    total_rows = 0
    for (table_name,) in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cursor.fetchone()[0]
        total_rows += count
    
    db_size = TARGET_DB_PATH.stat().st_size / (1024 * 1024)
    
    print(f"\nSDM Database Info:")
    print(f"  Location: {TARGET_DB_PATH}")
    print(f"  Size: {db_size:.2f} MB")
    print(f"  Tables: {len(tables)}")
    print(f"  Total Rows: {total_rows:,}\n")


In [4]:
# Schema Mirroring Functions

def create_table_from_schema(conn: sqlite3.Connection, table_name: str, schema: Dict[str, Any], include_fk: bool = False):
    """Create table in SDM matching source schema exactly."""
    cursor = conn.cursor()
    
    # Drop existing table if it exists
    cursor.execute(f"DROP TABLE IF EXISTS {table_name}")
    
    # Build CREATE TABLE statement
    columns_sql = []
    pk_columns = []
    
    for col_info in schema['columns']:
        col_sql = f"{col_info['name']} {col_info['type']}"
        
        if col_info['not_null']:
            col_sql += " NOT NULL"
        
        if col_info['default'] is not None:
            col_sql += f" DEFAULT {col_info['default']}"
        
        columns_sql.append(col_sql)
        
        if col_info['pk']:
            pk_columns.append(col_info['name'])
    
    # Add primary key constraint if exists
    if pk_columns:
        columns_sql.append(f"PRIMARY KEY ({', '.join(pk_columns)})")
    
    # Add foreign key constraints only if requested (after data load)
    if include_fk:
        for fk in schema['foreign_keys']:
            fk_sql = f"FOREIGN KEY ({fk['from_column']}) REFERENCES {fk['to_table']}({fk['to_column']})"
            if fk['on_delete']:
                fk_sql += f" ON DELETE {fk['on_delete']}"
            if fk['on_update']:
                fk_sql += f" ON UPDATE {fk['on_update']}"
            columns_sql.append(fk_sql)
    
    create_sql = f"CREATE TABLE {table_name} (\n  " + ",\n  ".join(columns_sql) + "\n)"
    
    try:
        cursor.execute(create_sql)
        conn.commit()
    except Exception as e:
        logger.error(f"Error creating table {table_name}: {e}")
        logger.error(f"SQL: {create_sql}")
        raise
    
    logger.info(f"Created table {table_name}")


def mirror_sqlite_schema(sdm_conn: sqlite3.Connection, source_db_path: Path, table_prefix: str = "", include_fk: bool = False):
    """Mirror all tables from source SQLite database to SDM."""
    
    logger.info(f"Mirroring schema from {source_db_path}")
    
    schema = get_sqlite_schema(source_db_path)
    
    for table_name, table_schema in schema.items():
        target_table_name = f"{table_prefix}{table_name}" if table_prefix else table_name
        create_table_from_schema(sdm_conn, target_table_name, table_schema, include_fk=include_fk)
    
    logger.info(f"Schema mirroring complete for {source_db_path}")


def create_csv_table(conn: sqlite3.Connection, table_name: str, csv_path: Path):
    """Create table for CSV data with inferred types and composite PK."""
    cursor = conn.cursor()
    
    # Drop existing table if it exists
    cursor.execute(f"DROP TABLE IF EXISTS {table_name}")
    
    # Detect encoding
    encoding = detect_csv_encoding(csv_path)
    logger.info(f"  CSV encoding detected: {encoding}")
    
    # Infer column types
    inferred_types = infer_csv_types(csv_path)
    
    # Get column names from CSV header
    with open(csv_path, 'r', encoding=encoding) as f:
        reader = csv.DictReader(f)
        column_names = reader.fieldnames
    
    # Build column definitions
    columns_sql = []
    for col_name in column_names:
        col_type = inferred_types.get(col_name, 'TEXT')
        columns_sql.append(f"{col_name} {col_type}")
    
    # Add composite primary key
    pk_cols = CSV_PRIMARY_KEYS.get(table_name, [])
    if pk_cols:
        columns_sql.append(f"PRIMARY KEY ({', '.join(pk_cols)})")
    
    create_sql = f"CREATE TABLE {table_name} (\n  " + ",\n  ".join(columns_sql) + "\n)"
    
    cursor.execute(create_sql)
    conn.commit()
    
    logger.info(f"Created CSV table {table_name}")


In [5]:
# Data Loading Functions - WITH BETTER ERROR HANDLING

class DeferredForeignKeyLoad(Exception):
    """Raised when a table must be retried after dependent FK tables are loaded."""


def load_sqlite_table_incremental(
    sdm_conn: sqlite3.Connection,
    source_db_path: Path,
    source_table_name: str,
    target_table_name: str
) -> Tuple[int, int, int]:
    """
    Incrementally load SQLite table data.
    Returns: (inserted, updated, deleted)
    """
    
    # Connect to source database
    source_conn = sqlite3.connect(source_db_path)
    source_conn.row_factory = sqlite3.Row
    source_cursor = source_conn.cursor()
    
    # Get source schema to extract primary key
    schema = get_sqlite_schema(source_db_path)
    table_schema = schema.get(source_table_name, {})
    pk_columns = table_schema.get('primary_key', [])
    
    # Get stored hashes
    stored_hashes = get_row_hashes(sdm_conn, target_table_name)
    current_hashes = {}
    
    sdm_cursor = sdm_conn.cursor()
    inserted = updated = deleted = 0
    
    try:
        # Process current data from source
        source_cursor.execute(f"SELECT * FROM {source_table_name}")
        rows = source_cursor.fetchall()
        logger.info(f"    Source table {source_table_name} has {len(rows)} rows, PK: {pk_columns}")
        
        for row_idx, row in enumerate(rows):
            row_dict = dict(row)
            row_hash = compute_row_hash(row_dict)
            
            if pk_columns:
                pk_tuple = extract_primary_key_tuple(row_dict, pk_columns)
            else:
                # No PK - use all columns as key  
                pk_tuple = None
            
            current_hashes[pk_tuple] = row_hash
            
            # Check if row exists and has changed
            if pk_tuple and pk_tuple in stored_hashes:
                if stored_hashes[pk_tuple] != row_hash:
                    # Row modified - UPDATE
                    set_clause = ", ".join([f"{k} = ?" for k in row_dict.keys()])
                    where_clause = " AND ".join([f"{k} = ?" for k in pk_columns])
                    update_sql = f"UPDATE {target_table_name} SET {set_clause} WHERE {where_clause}"
                    
                    values = list(row_dict.values()) + [row_dict[k] for k in pk_columns]
                    sdm_cursor.execute(update_sql, values)
                    updated += 1
            else:
                # New row - INSERT
                cols = ", ".join(row_dict.keys())
                placeholders = ", ".join(["?" for _ in row_dict])
                insert_sql = f"INSERT INTO {target_table_name} ({cols}) VALUES ({placeholders})"
                
                try:
                    sdm_cursor.execute(insert_sql, tuple(row_dict.values()))
                    inserted += 1
                except sqlite3.OperationalError as insert_error:
                    message = str(insert_error).lower()
                    if "no such table" in message:
                        logger.info(
                            f"      Deferring {target_table_name} because a referenced table is not ready: {insert_error}"
                        )
                        sdm_conn.rollback()
                        raise DeferredForeignKeyLoad(str(insert_error)) from insert_error
                    logger.error(f"      INSERT error on row {row_idx} for {target_table_name}: {insert_error}")
                    logger.error(f"      SQL: {insert_sql}")
                    logger.error(f"      Values types: {[type(v).__name__ for v in row_dict.values()]}")
                except sqlite3.IntegrityError as ie:
                    # Handle integrity errors gracefully - they might be FK issues that are OK
                    if "FOREIGN KEY constraint failed" in str(ie):
                        logger.debug(f"      Skipping row {row_idx} due to FK constraint: {ie}")
                        # Continue without inserting this row
                    else:
                        logger.error(f"      Integrity error on row {row_idx}: {ie}")
                        logger.error(f"      SQL: {insert_sql}")
                        logger.error(f"      Values: {tuple(row_dict.values())}")
                        # Don't raise - continue with next row
                except Exception as insert_error:
                    logger.error(f"      INSERT error on row {row_idx} for {target_table_name}: {insert_error}")
                    logger.error(f"      SQL: {insert_sql}")
                    logger.error(f"      Values types: {[type(v).__name__ for v in row_dict.values()]}")
                    # Don't raise - continue with next row
        
        # Handle deletions - find rows in stored but not in current
        if pk_columns:
            for stored_pk in stored_hashes.keys():
                if stored_pk not in current_hashes:
                    # Row deleted - DELETE
                    where_clause = " AND ".join([f"{k} = ?" for k in pk_columns])
                    delete_sql = f"DELETE FROM {target_table_name} WHERE {where_clause}"
                    try:
                        sdm_cursor.execute(delete_sql, stored_pk)
                        deleted += 1
                    except Exception as del_error:
                        logger.warning(f"      Delete error: {del_error}")
        
        # Save updated hashes
        save_row_hashes(sdm_conn, target_table_name, current_hashes, 'sqlite')
        sdm_conn.commit()
        logger.info(f"    Committed changes: +{inserted} ~{updated} -{deleted}")
        
    except DeferredForeignKeyLoad:
        raise
    except Exception as e:
        sdm_conn.rollback()
        logger.error(f"Error loading {target_table_name}: {e}", exc_info=True)
        # Don't raise - just log and continue
    finally:
        source_conn.close()
    
    logger.info(f"  {target_table_name}: +{inserted} ~{updated} -{deleted}")
    return inserted, updated, deleted


def load_csv_incremental(
    sdm_conn: sqlite3.Connection,
    csv_path: Path,
    table_name: str
) -> Tuple[int, int, int]:
    """
    Incrementally load CSV data.
    Returns: (inserted, updated, deleted)
    """
    
    # Detect encoding
    encoding = detect_csv_encoding(csv_path)
    
    pk_columns = CSV_PRIMARY_KEYS.get(table_name, [])
    stored_hashes = get_row_hashes(sdm_conn, table_name)
    current_hashes = {}
    
    sdm_cursor = sdm_conn.cursor()
    inserted = updated = deleted = 0
    
    try:
        # Process current data from CSV
        with open(csv_path, 'r', encoding=encoding) as f:
            reader = csv.DictReader(f)
            row_count = 0
            
            for row in reader:
                row_count += 1
                # Clean up row (remove extra whitespace)
                row_dict = {k: v.strip() if isinstance(v, str) else v for k, v in row.items()}
                row_hash = compute_row_hash(row_dict)
                
                if pk_columns:
                    pk_tuple = extract_primary_key_tuple(row_dict, pk_columns)
                else:
                    pk_tuple = None
                
                current_hashes[pk_tuple] = row_hash
                
                # Check if row exists and has changed
                if pk_tuple and pk_tuple in stored_hashes:
                    if stored_hashes[pk_tuple] != row_hash:
                        # Row modified - UPDATE
                        set_clause = ", ".join([f"{k} = ?" for k in row_dict.keys()])
                        where_clause = " AND ".join([f"{k} = ?" for k in pk_columns])
                        update_sql = f"UPDATE {table_name} SET {set_clause} WHERE {where_clause}"
                        
                        values = list(row_dict.values()) + [row_dict[k] for k in pk_columns]
                        sdm_cursor.execute(update_sql, values)
                        updated += 1
                else:
                    # New row - INSERT
                    cols = ", ".join(row_dict.keys())
                    placeholders = ", ".join(["?" for _ in row_dict])
                    insert_sql = f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})"
                    try:
                        sdm_cursor.execute(insert_sql, tuple(row_dict.values()))
                        inserted += 1
                    except Exception as e:
                        logger.warning(f"      Skip row {row_count}: {e}")
        
        # Handle deletions
        if pk_columns:
            for stored_pk in stored_hashes.keys():
                if stored_pk not in current_hashes:
                    # Row deleted - DELETE
                    where_clause = " AND ".join([f"{k} = ?" for k in pk_columns])
                    delete_sql = f"DELETE FROM {table_name} WHERE {where_clause}"
                    try:
                        sdm_cursor.execute(delete_sql, stored_pk)
                        deleted += 1
                    except Exception as e:
                        logger.warning(f"      Delete error: {e}")
        
        # Save updated hashes
        save_row_hashes(sdm_conn, table_name, current_hashes, 'csv')
        sdm_conn.commit()
        
    except Exception as e:
        logger.error(f"Error loading {table_name}: {e}", exc_info=True)
    
    logger.info(f"  {table_name}: +{inserted} ~{updated} -{deleted}")
    return inserted, updated, deleted


In [6]:
def rebuild_sqlite_tables_with_foreign_keys(sdm_conn: sqlite3.Connection) -> None:
    """Rebuild loaded SQLite-backed tables so the final SDM contains FK constraints."""

    logger.info("Rebuilding SQLite tables with foreign keys enabled")
    sdm_conn.execute("PRAGMA foreign_keys = OFF")

    for source_filename, table_prefix in SQLITE_SOURCES:
        source_path = GO_DIR / source_filename
        if not source_path.exists():
            continue

        source_schema = get_sqlite_schema(source_path)
        for source_table, table_schema in source_schema.items():
            target_table_name = f"{table_prefix}{source_table}" if table_prefix else source_table

            cursor = sdm_conn.cursor()
            cursor.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name = ?", (target_table_name,))
            if not cursor.fetchone():
                continue

            temp_table_name = f"{target_table_name}__fk_tmp"
            cursor.execute(f"DROP TABLE IF EXISTS {temp_table_name}")

            create_table_from_schema(sdm_conn, temp_table_name, table_schema, include_fk=True)

            column_names = [col['name'] for col in table_schema['columns']]
            column_list = ", ".join(column_names)
            cursor.execute(
                f"INSERT INTO {temp_table_name} ({column_list}) SELECT {column_list} FROM {target_table_name}"
            )

            cursor.execute(f"DROP TABLE {target_table_name}")
            cursor.execute(f"ALTER TABLE {temp_table_name} RENAME TO {target_table_name}")
            sdm_conn.commit()

    sdm_conn.execute("PRAGMA foreign_keys = ON")
    sdm_conn.commit()
    logger.info("Foreign key rebuild complete")


In [7]:
def load_sdm_parallel(force_recreate: bool = False) -> Dict[str, Any]:
    """
    Load SDM with sequential processing.
    Supports hash-based incremental loading and preserves all foreign keys.
    """
    
    sdm_conn = initialize_sdm_database(force_recreate=force_recreate)
    sdm_conn.execute("PRAGMA foreign_keys = ON")
    
    initial_load = is_initial_load(sdm_conn)
    load_type_str = 'INITIAL' if initial_load else 'INCREMENTAL'
    logger.info(f"Load type: {load_type_str}")
    print(f"Load type: {load_type_str}\n")
    
    start_time = time.time()
    results = {}
    
    try:
        # Disable FK constraints during load
        sdm_conn.execute("PRAGMA foreign_keys = OFF")
        
        # Load SQLite sources
        for source_filename, table_prefix in SQLITE_SOURCES:
            source_path = GO_DIR / source_filename
            if not source_path.exists():
                logger.warning(f"Source not found: {source_path}")
                print(f"⚠ Source not found: {source_path}")
                continue
            
            logger.info(f"Processing SQLite: {source_filename}")
            print(f"📂 SQLite: {source_filename}")
            
            try:
                # Mirror schema
                logger.info(f"  Mirroring schema...")
                mirror_sqlite_schema(sdm_conn, source_path, table_prefix)
                logger.info(f"  Schema mirrored")
                
                # Get tables and load data
                schema = get_sqlite_schema(source_path)
                pending_tables = [table_name for table_name in schema.keys() if not table_name.startswith('sqlite_')]
                logger.info(f"  Found {len(pending_tables)} tables")
                print(f"  Tables: {len(pending_tables)}")
                
                stalled_tables = 0
                while pending_tables:
                    table_name = pending_tables.pop(0)
                    target_table_name = f"{table_prefix}{table_name}"
                    logger.info(f"    Loading table: {source_filename}.{table_name} -> {target_table_name}")
                    
                    record_source_metadata(sdm_conn, target_table_name, 'sqlite')
                    
                    try:
                        inserted, updated, deleted = load_sqlite_table_incremental(
                            sdm_conn, source_path, table_name, target_table_name
                        )
                        results[target_table_name] = (inserted, updated, deleted)
                        print(f"    {table_name:40} +{inserted:6} ~{updated:6} -{deleted:6}")
                        stalled_tables = 0
                    except DeferredForeignKeyLoad as deferred_error:
                        pending_tables.append(table_name)
                        stalled_tables += 1
                        logger.info(f"    Re-queuing {target_table_name} until FK dependency is available: {deferred_error}")
                        print(f"    {table_name:40} deferred, moved to end of queue")
                        
                        if stalled_tables >= len(pending_tables):
                            logger.error(
                                f"Unable to make progress while loading {source_filename}; remaining tables still depend on missing foreign keys"
                            )
                            print(f"    ⚠ Stopped retrying {source_filename} because no table could progress")
                            break
                    except Exception as table_error:
                        logger.error(f"    Error loading {table_name}: {table_error}")
                        print(f"    {table_name:40} ERROR: {table_error}")
                    
            except Exception as e:
                logger.error(f"Error processing {source_filename}: {e}", exc_info=True)
                print(f"❌ Error: {e}")
        
        # Load CSV sources
        for source_filename, table_name in CSV_SOURCES:
            source_path = GO_DIR / source_filename
            if not source_path.exists():
                logger.warning(f"CSV not found: {source_path}")
                print(f"⚠ CSV not found: {source_path}")
                continue
            
            logger.info(f"Processing CSV: {source_filename}")
            print(f"📄 CSV: {source_filename}")
            
            try:
                # Create table
                logger.info(f"  Creating table: {table_name}")
                create_csv_table(sdm_conn, table_name, source_path)
                record_source_metadata(sdm_conn, table_name, 'csv')
                
                # Load data
                logger.info(f"  Loading data...")
                inserted, updated, deleted = load_csv_incremental(sdm_conn, source_path, table_name)
                results[table_name] = (inserted, updated, deleted)
                print(f"  {table_name:40} +{inserted:6} ~{updated:6} -{deleted:6}")
                
            except Exception as e:
                logger.error(f"Error processing {source_filename}: {e}", exc_info=True)
                print(f"❌ Error: {e}")
        
        # Re-enable FK constraints
        sdm_conn.execute("PRAGMA foreign_keys = ON")
        
        # Rebuild SQLite tables so the final SDM preserves foreign key constraints
        rebuild_sqlite_tables_with_foreign_keys(sdm_conn)
        
        # Validate FK integrity
        logger.info("Validating foreign key integrity...")
        orphaned = validate_foreign_keys(sdm_conn)
        if orphaned:
            logger.warning(f"Found {len(orphaned)} orphaned FK references")
        
    finally:
        sdm_conn.close()
    
    elapsed = time.time() - start_time
    
    return {
        'status': 'success',
        'load_type': 'initial' if initial_load else 'incremental',
        'elapsed_seconds': elapsed,
        'tables_loaded': len(results),
        'table_results': results
    }


In [8]:
def validate_foreign_keys(sdm_conn: sqlite3.Connection) -> Dict[str, List[Dict[str, Any]]]:
    """
    Validate foreign key constraints in SDM.
    Returns any orphaned records (FKs pointing to non-existent records).
    """
    
    issues = {}
    cursor = sdm_conn.cursor()
    
    # Get all tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' AND name != 'sdm_load_metadata'")
    tables = cursor.fetchall()
    
    for (table_name,) in tables:
        try:
            # Get foreign keys for this table
            cursor.execute(f"PRAGMA foreign_key_list({table_name})")
            fks = cursor.fetchall()
            
            for fk_id, seq, ref_table, from_col, ref_col, on_delete, on_update, match in fks:
                # Check if reference table exists
                cursor.execute(f"SELECT 1 FROM sqlite_master WHERE type='table' AND name = ?", (ref_table,))
                if not cursor.fetchone():
                    issues[f"{table_name}.{from_col} -> {ref_table}.{ref_col}"] = [{'error': 'Reference table does not exist'}]
                    continue
                
                # Check for orphaned records
                orphaned_query = f"""
                    SELECT {from_col}, COUNT(*) as count
                    FROM {table_name} t
                    WHERE {from_col} IS NOT NULL
                    AND NOT EXISTS (
                        SELECT 1 FROM {ref_table} r WHERE r.{ref_col} = t.{from_col}
                    )
                    GROUP BY {from_col}
                """
                
                cursor.execute(orphaned_query)
                orphaned = cursor.fetchall()
                
                if orphaned:
                    key = f"{table_name}.{from_col} -> {ref_table}.{ref_col}"
                    issues[key] = [{'value': row[0], 'count': row[1]} for row in orphaned]
        except Exception as e:
            logger.error(f"Error validating FKs for {table_name}: {e}")
    
    return issues


In [9]:
# EXECUTE: SDM Load with Parallel Processing

import gc

# Close any existing connections
try:
    sdm_conn.close()
except:
    pass

gc.collect()

print("Starting SDM (Source Data Model) Load Process...\n")

# Run the load with force_recreate to ensure fresh load
load_result = load_sdm_parallel(force_recreate=True)

# Display report
print("\n" + generate_sdm_report(load_result))

# Open connection for verification
sdm_conn = sqlite3.connect(TARGET_DB_PATH)
sdm_conn.execute("PRAGMA foreign_keys = ON")

# Display SDM information
display_sdm_info(sdm_conn)

# Check if tables were created
cursor = sdm_conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' AND name != 'sdm_load_metadata'")
tables_exist = cursor.fetchall()

if not tables_exist:
    print("⚠️  WARNING: No data tables found in SDM. Check load process logs above.")
    print("Debugging info - checking metadata:")
    cursor.execute("SELECT source_name, source_type, row_count FROM sdm_load_metadata ORDER BY source_name")
    metadata = cursor.fetchall()
    for row in metadata:
        print(f"  {row[0]:50} | {row[1]:6} | {row[2]}")
else:
    # Verify schema
    print("Verifying Schema Integrity...\n")
    verification = verify_sdm_schema(sdm_conn)
    
    print("Schema Verification Results:")
    print("-" * 80)
    schema_issues = [name for name, match in verification['schema_matches'].items() if match is False]
    if schema_issues:
        print(f"⚠️  Schema mismatches found in {len(schema_issues)} tables:")
        for table in schema_issues:
            print(f"   - {table}")
    else:
        print("✓ All schemas match source databases")
    
    print("\n")
    
    # Check foreign key integrity
    print("Validating Foreign Key Integrity...\n")
    orphaned = validate_foreign_keys(sdm_conn)
    
    if orphaned:
        print(f"⚠️  Found orphaned foreign key references:")
        for fk_name, issues in orphaned.items():
            print(f"   {fk_name}: {len(issues)} issue(s)")
            for issue in issues[:5]:  # Show first 5
                if 'error' in issue:
                    print(f"      - Error: {issue['error']}")
                else:
                    print(f"      - Value: {issue['value']} (count: {issue['count']})")
    else:
        print("✓ All foreign keys are valid")

print("\n" + "=" * 80)
print("SDM Load Complete!")
print("=" * 80 + "\n")

sdm_conn.close()


2026-05-04 15:59:33,779 - INFO - Load type: INITIAL
2026-05-04 15:59:33,780 - WARNING - Source not found: GreatOutdoors\CRM-data.sqlite
2026-05-04 15:59:33,781 - WARNING - Source not found: GreatOutdoors\GO_SALES-data.sqlite
2026-05-04 15:59:33,782 - WARNING - Source not found: GreatOutdoors\GO_STAFF-data.sqlite
2026-05-04 15:59:33,782 - WARNING - CSV not found: GreatOutdoors\INVENTORY_LEVELS-data.csv
2026-05-04 15:59:33,783 - WARNING - CSV not found: GreatOutdoors\PRODUCT_FORECAST-data.csv
2026-05-04 15:59:33,784 - WARNING - CSV not found: GreatOutdoors\SALES_TARGET-data.csv
2026-05-04 15:59:33,785 - INFO - Rebuilding SQLite tables with foreign keys enabled
2026-05-04 15:59:33,786 - INFO - Foreign key rebuild complete
2026-05-04 15:59:33,786 - INFO - Validating foreign key integrity...


Starting SDM (Source Data Model) Load Process...

Load type: INITIAL

⚠ Source not found: GreatOutdoors\CRM-data.sqlite
⚠ Source not found: GreatOutdoors\GO_SALES-data.sqlite
⚠ Source not found: GreatOutdoors\GO_STAFF-data.sqlite
⚠ CSV not found: GreatOutdoors\INVENTORY_LEVELS-data.csv
⚠ CSV not found: GreatOutdoors\PRODUCT_FORECAST-data.csv
⚠ CSV not found: GreatOutdoors\SALES_TARGET-data.csv


╔════════════════════════════════════════════════════════════════╗
║              SDM LOAD REPORT                                   ║
╚════════════════════════════════════════════════════════════════╝

Load Type:        INITIAL
Status:           SUCCESS
Elapsed Time:     0.01 seconds
Tables Processed: 0

SUMMARY BY TABLE:
----------------------------------------------------------------------
----------------------------------------------------------------------
TOTAL                                         +     0 ~     0 -     0



SDM Database Info:
  Location: GreatOutdoors_SDM.db
  Size: 0.

In [10]:
# UTILITIES: Helper Functions for Testing and Debugging

def reset_sdm_database():
    """Delete and recreate the SDM database (for testing)."""
    if TARGET_DB_PATH.exists():
        TARGET_DB_PATH.unlink()
        logger.info(f"Deleted SDM database")
    print("SDM database reset. Ready for fresh load.")


def list_sdm_tables():
    """List all tables in SDM with row counts and column counts."""
    conn = sqlite3.connect(TARGET_DB_PATH)
    cursor = conn.cursor()
    
    print("\nSDM Tables:")
    print("-" * 80)
    
    cursor.execute("""
        SELECT name FROM sqlite_master 
        WHERE type='table' AND name NOT LIKE 'sqlite_%'
        ORDER BY name
    """)
    tables = cursor.fetchall()
    
    for (table_name,) in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cursor.fetchone()[0]
        
        cursor.execute(f"PRAGMA table_info({table_name})")
        cols = cursor.fetchall()
        
        cursor.execute(f"PRAGMA foreign_key_list({table_name})")
        fks = cursor.fetchall()
        
        fk_str = f" | FKs: {len(fks)}" if fks else ""
        print(f"{table_name:45} | {len(cols):2} cols | {count:8} rows{fk_str}")
    
    conn.close()


def query_sdm(sql: str):
    """Execute a query against SDM database."""
    conn = sqlite3.connect(TARGET_DB_PATH)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    cursor.execute(sql)
    rows = cursor.fetchall()
    
    print(f"\nQuery Results ({len(rows)} rows):")
    print("-" * 80)
    
    if rows:
        # Print header
        headers = [description[0] for description in cursor.description]
        print(" | ".join(f"{h:20}" for h in headers))
        print("-" * 80)
        
        # Print rows (limit to 20)
        for i, row in enumerate(rows[:20]):
            print(" | ".join(f"{str(row[h])[:20]:20}" for h in headers))
        
        if len(rows) > 20:
            print(f"... and {len(rows) - 20} more rows")
    
    conn.close()


def compare_row_counts():
    """Compare row counts between SDM and source databases."""
    
    print("\nRow Count Comparison:")
    print("-" * 80)
    
    sdm_conn = sqlite3.connect(TARGET_DB_PATH)
    sdm_cursor = sdm_conn.cursor()
    
    # SQLite sources
    for source_filename, table_prefix in SQLITE_SOURCES:
        source_path = GO_DIR / source_filename
        
        if not source_path.exists():
            continue
        
        print(f"\n{source_filename}:")
        
        source_conn = sqlite3.connect(source_path)
        source_cursor = source_conn.cursor()
        
        source_cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables = source_cursor.fetchall()
        
        for (table_name,) in tables:
            if table_name.startswith('sqlite_'):
                continue
            
            source_cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
            source_count = source_cursor.fetchone()[0]
            
            target_table = f"{table_prefix}{table_name}"
            sdm_cursor.execute(f"SELECT COUNT(*) FROM {target_table}")
            sdm_count = sdm_cursor.fetchone()[0]
            
            match = "✓" if source_count == sdm_count else "✗"
            print(f"  {table_name:40} | Source: {source_count:8} | SDM: {sdm_count:8} {match}")
        
        source_conn.close()
    
    # CSV sources
    print(f"\nCSV Sources:")
    for source_filename, table_name in CSV_SOURCES:
        source_path = GO_DIR / source_filename
        
        if not source_path.exists():
            continue
        
        # Count rows in CSV
        csv_count = 0
        with open(source_path, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            csv_count = sum(1 for _ in reader)
        
        sdm_cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        sdm_count = sdm_cursor.fetchone()[0]
        
        match = "✓" if csv_count == sdm_count else "✗"
        print(f"  {table_name:40} | Source: {csv_count:8} | SDM: {sdm_count:8} {match}")
    
    sdm_conn.close()


print("Utility functions loaded. Available functions:")
print("  - reset_sdm_database()          : Delete and recreate SDM (for testing)")
print("  - list_sdm_tables()             : List all tables with row/col counts")
print("  - query_sdm(sql)                : Execute query against SDM")
print("  - compare_row_counts()          : Compare source vs SDM row counts")


Utility functions loaded. Available functions:
  - reset_sdm_database()          : Delete and recreate SDM (for testing)
  - list_sdm_tables()             : List all tables with row/col counts
  - query_sdm(sql)                : Execute query against SDM
  - compare_row_counts()          : Compare source vs SDM row counts


In [11]:
# FINAL VERIFICATION: SDM Database Summary

print("\n" + "=" * 90)
print("SDM DATABASE SUMMARY - FINAL VERIFICATION")
print("=" * 90 + "\n")

import sqlite3

sdm_path = ROOT / "GreatOutdoors_SDM.db"
conn = sqlite3.connect(sdm_path)
cursor = conn.cursor()

print("DATA TABLES IN SDM:")
print("-" * 90)

total_rows = 0
cursor.execute("""
    SELECT name FROM sqlite_master 
    WHERE type='table' AND name NOT LIKE 'sqlite_%' AND name != 'sdm_load_metadata'
    ORDER BY name
""")
tables = cursor.fetchall()

for (table_name,) in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    count = cursor.fetchone()[0]
    total_rows += count
    
    cursor.execute(f"PRAGMA table_info({table_name})")
    cols = cursor.fetchall()
    col_count = len(cols)
    
    cursor.execute(f"PRAGMA foreign_key_list({table_name})")
    fks = cursor.fetchall()
    fk_count = len(fks)
    
    fk_str = f" | FK: {fk_count}" if fks else ""
    print(f"{table_name:45} | {col_count:3} cols | {count:8} rows{fk_str}")

print("-" * 90)
print(f"{'TOTAL':45} | {'-':>3}     | {total_rows:8} rows")

print("\n✅ SDM Database successfully created!")
print(f"   Database file: {sdm_path}")
print(f"   Database size: {sdm_path.stat().st_size / (1024*1024):.2f} MB")
print(f"   Total data tables: {len(tables)}")
print(f"   Total rows: {total_rows:,}")

print("\n" + "=" * 90 + "\n")

conn.close()



SDM DATABASE SUMMARY - FINAL VERIFICATION

DATA TABLES IN SDM:
------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------
TOTAL                                         |   -     |        0 rows

✅ SDM Database successfully created!
   Database file: GreatOutdoors_SDM.db
   Database size: 0.01 MB
   Total data tables: 0
   Total rows: 0




In [12]:
# DIAGNOSTIC: Check Source Databases and Debug

print("Diagnostic: Checking source databases\n")
print("=" * 80)

# Check SQLite sources
for source_filename, table_prefix in SQLITE_SOURCES:
    source_path = GO_DIR / source_filename
    print(f"\n{source_filename}:")
    print(f"  Exists: {source_path.exists()}")
    
    if source_path.exists():
        try:
            schema = get_sqlite_schema(source_path)
            print(f"  Tables: {len(schema)}")
            for table_name in list(schema.keys())[:5]:
                print(f"    - {table_name}")
        except Exception as e:
            print(f"  Error reading schema: {e}")

# Check CSV sources
print("\n")
for source_filename, table_name in CSV_SOURCES:
    source_path = GO_DIR / source_filename
    print(f"\n{source_filename}:")
    print(f"  Exists: {source_path.exists()}")
    
    if source_path.exists():
        try:
            with open(source_path, 'r') as f:
                reader = csv.DictReader(f)
                cols = reader.fieldnames
                count = sum(1 for _ in reader)
            print(f"  Columns: {len(cols)}")
            print(f"  Rows: {count}")
            print(f"  Column names: {cols[:3]}...")
        except Exception as e:
            print(f"  Error reading CSV: {e}")

print("\n" + "=" * 80)


Diagnostic: Checking source databases


CRM-data.sqlite:
  Exists: False

GO_SALES-data.sqlite:
  Exists: False

GO_STAFF-data.sqlite:
  Exists: False



INVENTORY_LEVELS-data.csv:
  Exists: False

PRODUCT_FORECAST-data.csv:
  Exists: False

SALES_TARGET-data.csv:
  Exists: False



In [13]:
# DEBUG: Test data extraction from source

print("Testing data extraction from CRM database...\n")

crm_path = GO_DIR / "CRM-data.sqlite"
source_conn = sqlite3.connect(crm_path)
source_conn.row_factory = sqlite3.Row
cursor = source_conn.cursor()

# Get first table
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' LIMIT 1")
table_name = cursor.fetchone()[0]
print(f"Testing table: {table_name}\n")

# Get schema
cursor.execute(f"PRAGMA table_info({table_name})")
columns = cursor.fetchall()
print(f"Columns: {[col[1] for col in columns]}\n")

# Get first 3 rows
cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
rows = cursor.fetchall()

print(f"Found {len(rows)} rows")
for i, row in enumerate(rows):
    row_dict = dict(row)
    print(f"\nRow {i+1}: {row_dict}")

source_conn.close()

print("\n" + "=" * 80)


Testing data extraction from CRM database...



OperationalError: unable to open database file

In [ ]:
# DEBUG: Test CREATE TABLE SQL generation

print("Testing SQL generation for first table...\n")

crm_path = GO_DIR / "CRM-data.sqlite"
schema = get_sqlite_schema(crm_path)

first_table = list(schema.keys())[0]
table_schema = schema[first_table]

print(f"Table: {first_table}")
print(f"Columns: {[col['name'] for col in table_schema['columns']]}")
print(f"Primary Key: {table_schema['primary_key']}")
print(f"Foreign Keys: {table_schema['foreign_keys']}")

# Build CREATE TABLE statement manually to see what's wrong
columns_sql = []
pk_columns = []

for col_info in table_schema['columns']:
    col_sql = f"{col_info['name']} {col_info['type']}"
    
    if col_info['not_null']:
        col_sql += " NOT NULL"
    
    if col_info['default'] is not None:
        col_sql += f" DEFAULT {col_info['default']}"
    
    columns_sql.append(col_sql)
    
    if col_info['pk']:
        pk_columns.append(col_info['name'])

# Add primary key constraint if exists
if pk_columns:
    columns_sql.append(f"PRIMARY KEY ({', '.join(pk_columns)})")

# Add foreign key constraints
for fk in table_schema['foreign_keys']:
    fk_sql = f"FOREIGN KEY ({fk['from_column']}) REFERENCES {fk['to_table']}({fk['to_column']})"
    if fk['on_delete']:
        fk_sql += f" ON DELETE {fk['on_delete']}"
    if fk['on_update']:
        fk_sql += f" ON UPDATE {fk['on_update']}"
    columns_sql.append(fk_sql)

create_sql = f"CREATE TABLE {first_table} (\n  " + ",\n  ".join(columns_sql) + "\n)"

print("\nGenerated SQL:")
print(create_sql)

print("\n" + "=" * 80)


Testing SQL generation for first table...

Table: age_group
Columns: ['AGE_GROUP_CODE', 'UPPER_AGE', 'LOWER_AGE']
Primary Key: ['AGE_GROUP_CODE']
Foreign Keys: []

Generated SQL:
CREATE TABLE age_group (
  AGE_GROUP_CODE TEXT,
  UPPER_AGE TEXT,
  LOWER_AGE TEXT,
  PRIMARY KEY (AGE_GROUP_CODE)
)



In [ ]:
# DEBUG: Test INSERT into SDM

print("\nTesting direct INSERT into SDM...\n")

test_db_path = ROOT / "test_insert.db"
if test_db_path.exists():
    test_db_path.unlink()

test_conn = sqlite3.connect(test_db_path)

# Create a simple test table
test_conn.execute("""
    CREATE TABLE test_table (
        id INTEGER PRIMARY KEY,
        name TEXT,
        value INTEGER
    )
""")

# Insert test data
test_conn.execute("INSERT INTO test_table (name, value) VALUES (?, ?)", ("test1", 100))
test_conn.execute("INSERT INTO test_table (name, value) VALUES (?, ?)", ("test2", 200))
test_conn.commit()

# Read it back
cursor = test_conn.cursor()
cursor.execute("SELECT COUNT(*) FROM test_table")
count = cursor.fetchone()[0]
print(f"Inserted {count} rows into test_table")

test_conn.close()

# Delete test DB
test_db_path.unlink()

print("\n✓ Basic INSERT test passed")
print("=" * 80)



Testing direct INSERT into SDM...

Inserted 2 rows into test_table

✓ Basic INSERT test passed


In [ ]:
# Check which tables are empty

list_sdm_tables()



SDM Tables:
--------------------------------------------------------------------------------
CRM_age_group                                 |  3 cols |        6 rows
CRM_crm_country                               |  4 cols |       19 rows | FKs: 1
CRM_customer                                  |  4 cols |      109 rows | FKs: 2
CRM_customer_contact                          |  9 cols |      391 rows
CRM_customer_headquarters                     | 11 cols |      414 rows | FKs: 1
CRM_customer_segment                          |  4 cols |       12 rows
CRM_customer_store                            |  9 cols |      361 rows | FKs: 2
CRM_customer_type                             |  2 cols |        8 rows
CRM_sales_demographic                         |  4 cols |     2484 rows | FKs: 2
CRM_sales_territory                           |  2 cols |        5 rows
INVENTORY_LEVELS                              |  4 cols |     3888 rows
PRODUCT_FORECAST                              |  4 cols |     3872 ro

In [ ]:
# Find empty tables in SDM

import sqlite3

conn = sqlite3.connect(TARGET_DB_PATH)
cursor = conn.cursor()

cursor.execute("""
    SELECT name FROM sqlite_master 
    WHERE type='table' AND name NOT LIKE 'sqlite_%' AND name != 'sdm_load_metadata'
    ORDER BY name
""")
tables = cursor.fetchall()

empty_tables = []
loaded_tables = []

for (table_name,) in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    count = cursor.fetchone()[0]
    
    if count == 0:
        empty_tables.append(table_name)
    else:
        loaded_tables.append((table_name, count))

print("EMPTY TABLES (NO DATA LOADED):")
print("=" * 60)
for table in empty_tables:
    print(f"  ❌ {table}")

print(f"\nTOTAL EMPTY: {len(empty_tables)}")

print("\n\nTABLES WITH DATA:")
print("=" * 60)
for table, count in sorted(loaded_tables):
    print(f"  ✓ {table:45} ({count} rows)")

print(f"\nTOTAL LOADED: {len(loaded_tables)}")

conn.close()


EMPTY TABLES (NO DATA LOADED):

TOTAL EMPTY: 0


TABLES WITH DATA:
  ✓ CRM_age_group                                 (6 rows)
  ✓ CRM_crm_country                               (19 rows)
  ✓ CRM_customer                                  (109 rows)
  ✓ CRM_customer_contact                          (391 rows)
  ✓ CRM_customer_headquarters                     (414 rows)
  ✓ CRM_customer_segment                          (12 rows)
  ✓ CRM_customer_store                            (361 rows)
  ✓ CRM_customer_type                             (8 rows)
  ✓ CRM_sales_demographic                         (2484 rows)
  ✓ CRM_sales_territory                           (5 rows)
  ✓ INVENTORY_LEVELS                              (3888 rows)
  ✓ PRODUCT_FORECAST                              (3872 rows)
  ✓ SALES_TARGET                                  (39530 rows)
  ✓ SALES_country                                 (18 rows)
  ✓ SALES_order_details                           (40990 rows)
  ✓ SALES_order_head

In [ ]:
# Debug: Check source database table structure vs empty tables

print("SOURCE TABLE VERIFICATION")
print("=" * 80)

# Map empty tables back to their source
empty_mapped = {
    'CRM_crm_country': ('CRM-data.sqlite', 'crm_country'),
    'CRM_customer': ('CRM-data.sqlite', 'customer'),
    'CRM_customer_headquarters': ('CRM-data.sqlite', 'customer_headquarters'),
    'CRM_customer_store': ('CRM-data.sqlite', 'customer_store'),
    'CRM_sales_demographic': ('CRM-data.sqlite', 'sales_demographic'),
    'SALES_country': ('GO_SALES-data.sqlite', 'country'),
    'SALES_order_details': ('GO_SALES-data.sqlite', 'order_details'),
    'SALES_order_header': ('GO_SALES-data.sqlite', 'order_header'),
    'SALES_product': ('GO_SALES-data.sqlite', 'product'),
    'SALES_product_type': ('GO_SALES-data.sqlite', 'product_type'),
    'SALES_returned_item': ('GO_SALES-data.sqlite', 'returned_item'),
    'SALES_sales_branch': ('GO_SALES-data.sqlite', 'sales_branch'),
    'SALES_sales_staff': ('GO_SALES-data.sqlite', 'sales_staff'),
    'STAFF_course': ('GO_STAFF-data.sqlite', 'course'),
    'STAFF_sales_office': ('GO_STAFF-data.sqlite', 'sales_office'),
    'STAFF_sales_representative': ('GO_STAFF-data.sqlite', 'sales_representative'),
    'STAFF_satisfaction': ('GO_STAFF-data.sqlite', 'satisfaction'),
    'STAFF_satisfaction_type': ('GO_STAFF-data.sqlite', 'satisfaction_type'),
    'STAFF_training': ('GO_STAFF-data.sqlite', 'training'),
}

for sdm_table, (source_file, source_table) in empty_mapped.items():
    source_path = GO_DIR / source_file
    
    try:
        source_conn = sqlite3.connect(source_path)
        source_cursor = source_conn.cursor()
        
        # Check if table exists
        source_cursor.execute(f"SELECT COUNT(*) FROM {source_table}")
        count = source_cursor.fetchone()[0]
        
        # Get column info
        source_cursor.execute(f"PRAGMA table_info({source_table})")
        cols = source_cursor.fetchall()
        
        # Get FKs
        source_cursor.execute(f"PRAGMA foreign_key_list({source_table})")
        fks = source_cursor.fetchall()
        
        print(f"\n{sdm_table}:")
        print(f"  Source: {source_file}.{source_table}")
        print(f"  Rows: {count}")
        print(f"  Columns: {len(cols)}")
        print(f"  Foreign Keys: {len(fks)}")
        if fks:
            for fk in fks[:2]:
                print(f"    - {fk[3]} -> {fk[2]}.{fk[4]}")
        
        source_conn.close()
    except Exception as e:
        print(f"\n{sdm_table}:")
        print(f"  ERROR: {e}")


SOURCE TABLE VERIFICATION

CRM_crm_country:
  Source: CRM-data.sqlite.crm_country
  Rows: 19
  Columns: 4
  Foreign Keys: 1
    - SALES_TERRITORY_CODE -> sales_territory.SALES_TERRITORY_CODE

CRM_customer:
  Source: CRM-data.sqlite.customer
  Rows: 109
  Columns: 4
  Foreign Keys: 2
    - CUSTOMER_TYPE_CODE -> customer_type.CUSTOMER_TYPE_CODE
    - CUSTOMER_CODEMR -> customer_headquarters.CUSTOMER_CODEMR

CRM_customer_headquarters:
  Source: CRM-data.sqlite.customer_headquarters
  Rows: 414
  Columns: 11
  Foreign Keys: 1
    - SEGMENT_CODE -> customer_segment.SEGMENT_CODE

CRM_customer_store:
  Source: CRM-data.sqlite.customer_store
  Rows: 361
  Columns: 9
  Foreign Keys: 2
    - CUSTOMER_CODE -> customer.CUSTOMER_CODE
    - COUNTRY_CODE -> crm_country.COUNTRY_CODE

CRM_sales_demographic:
  Source: CRM-data.sqlite.sales_demographic
  Rows: 2484
  Columns: 4
  Foreign Keys: 2
    - CUSTOMER_CODEMR -> customer_headquarters.CUSTOMER_CODEMR
    - AGE_GROUP_CODE -> age_group.AGE_GROUP_COD

In [ ]:
# Debug: Check FK dependencies and issues

print("ANALYZING FOREIGN KEY DEPENDENCIES")
print("=" * 80)

# Check which tables have FK references and to what
crm_path = GO_DIR / "CRM-data.sqlite"
conn = sqlite3.connect(crm_path)
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

print("\nCRM Database FK Dependencies:")
print("-" * 80)

for (table_name,) in tables:
    cursor.execute(f"PRAGMA foreign_key_list({table_name})")
    fks = cursor.fetchall()
    
    if fks:
        print(f"\n{table_name}:")
        for fk in fks:
            fk_id, seq, ref_table, from_col, ref_col, on_delete, on_update, match = fk
            print(f"  {from_col} -> {ref_table}.{ref_col}")

conn.close()

# Now check the error - try inserting crm_country data directly
print("\n\n" + "=" * 80)
print("TESTING crm_country INSERTION")
print("=" * 80)

source_conn = sqlite3.connect(crm_path)
source_cursor = source_conn.cursor()

# Get first row from crm_country
source_cursor.execute("SELECT * FROM crm_country LIMIT 1")
col_names = [description[0] for description in source_cursor.description]
row = source_cursor.fetchone()

print(f"\ncrm_country has columns: {col_names}")
print(f"First row: {dict(zip(col_names, row))}")

source_conn.close()

# Now try inserting into SDM to see what error we get
sdm_conn = sqlite3.connect(TARGET_DB_PATH)
sdm_cursor = sdm_conn.cursor()

# Try to insert
try:
    cols_str = ", ".join(col_names)
    placeholders = ", ".join(["?" for _ in col_names])
    insert_sql = f"INSERT INTO CRM_crm_country ({cols_str}) VALUES ({placeholders})"
    
    print(f"\nTrying INSERT: {insert_sql[:80]}...")
    sdm_cursor.execute(insert_sql, row)
    sdm_conn.commit()
    print("✓ INSERT successful!")
except Exception as e:
    print(f"✗ INSERT failed: {e}")
    print(f"  Type: {type(e).__name__}")

sdm_conn.close()


ANALYZING FOREIGN KEY DEPENDENCIES

CRM Database FK Dependencies:
--------------------------------------------------------------------------------

customer_store:
  CUSTOMER_CODE -> customer.CUSTOMER_CODE
  COUNTRY_CODE -> crm_country.COUNTRY_CODE

crm_country:
  SALES_TERRITORY_CODE -> sales_territory.SALES_TERRITORY_CODE

customer:
  CUSTOMER_TYPE_CODE -> customer_type.CUSTOMER_TYPE_CODE
  CUSTOMER_CODEMR -> customer_headquarters.CUSTOMER_CODEMR

sales_demographic:
  CUSTOMER_CODEMR -> customer_headquarters.CUSTOMER_CODEMR
  AGE_GROUP_CODE -> age_group.AGE_GROUP_CODE

customer_headquarters:
  SEGMENT_CODE -> customer_segment.SEGMENT_CODE


TESTING crm_country INSERTION

crm_country has columns: ['COUNTRY_CODE', 'COUNTRY_EN', 'FLAG_IMAGE', 'SALES_TERRITORY_CODE']
First row: {'COUNTRY_CODE': '1', 'COUNTRY_EN': 'France', 'FLAG_IMAGE': 'F01', 'SALES_TERRITORY_CODE': '6'}

Trying INSERT: INSERT INTO CRM_crm_country (COUNTRY_CODE, COUNTRY_EN, FLAG_IMAGE, SALES_TERRITO...
✗ INSERT failed: 

In [ ]:
# DEBUG: Direct test of crm_country loading

print("DEBUGGING crm_country LOAD FAILURE")
print("=" * 80)

crm_path = GO_DIR / "CRM-data.sqlite"
test_sdm_path = ROOT / "test_debug.db"

if test_sdm_path.exists():
    test_sdm_path.unlink()

# Create test SDM
test_conn = sqlite3.connect(test_sdm_path)
test_conn.execute("PRAGMA foreign_keys = OFF")

# Get schema for crm_country
schema = get_sqlite_schema(crm_path)
crm_country_schema = schema.get('crm_country', {})
sales_territory_schema = schema.get('sales_territory', {})

print(f"\nCreating tables:")
print(f"  1. sales_territory (referenced by crm_country)")
create_table_from_schema(test_conn, 'sales_territory', sales_territory_schema)

print(f"  2. crm_country")
create_table_from_schema(test_conn, 'crm_country', crm_country_schema)

print(f"\nLoading data:")

# Load sales_territory first
source_conn = sqlite3.connect(crm_path)
source_conn.row_factory = sqlite3.Row
source_cursor = source_conn.cursor()

source_cursor.execute("SELECT * FROM sales_territory")
rows = source_cursor.fetchall()
print(f"\n  sales_territory: {len(rows)} rows")

for row in rows:
    row_dict = dict(row)
    cols = ", ".join(row_dict.keys())
    placeholders = ", ".join(["?" for _ in row_dict])
    sql = f"INSERT INTO sales_territory ({cols}) VALUES ({placeholders})"
    try:
        test_cursor = test_conn.cursor()
        test_cursor.execute(sql, tuple(row_dict.values()))
    except Exception as e:
        print(f"    ERROR on {row_dict}: {e}")

test_conn.commit()

# Now try crm_country
source_cursor.execute("SELECT * FROM crm_country")
rows = source_cursor.fetchall()
print(f"\n  crm_country: {len(rows)} rows to load")

success = 0
failed = 0

for row_idx, row in enumerate(rows):
    row_dict = dict(row)
    cols = ", ".join(row_dict.keys())
    placeholders = ", ".join(["?" for _ in row_dict])
    sql = f"INSERT INTO crm_country ({cols}) VALUES ({placeholders})"
    
    try:
        test_cursor = test_conn.cursor()
        test_cursor.execute(sql, tuple(row_dict.values()))
        success += 1
    except Exception as e:
        if failed < 3:  # Show first 3 errors
            print(f"    Row {row_idx}: {row_dict}")
            print(f"    ERROR: {e}\n")
        failed += 1

print(f"\n  Result: {success} inserted, {failed} failed")

test_conn.commit()

# Check counts
test_cursor = test_conn.cursor()
test_cursor.execute("SELECT COUNT(*) FROM crm_country")
count = test_cursor.fetchone()[0]
print(f"\n  crm_country now has: {count} rows")

source_conn.close()
test_conn.close()
test_sdm_path.unlink()

print("\n" + "=" * 80)


2026-05-03 22:38:33,329 - INFO - Created table sales_territory
2026-05-03 22:38:33,337 - INFO - Created table crm_country


DEBUGGING crm_country LOAD FAILURE

Creating tables:
  1. sales_territory (referenced by crm_country)
  2. crm_country

Loading data:

  sales_territory: 5 rows

  crm_country: 19 rows to load

  Result: 19 inserted, 0 failed

  crm_country now has: 19 rows



In [ ]:
# Test: Try to load crm_country and see the actual error

print("\nTesting crm_country load...")
print("=" * 60)

crm_path = GO_DIR / "CRM-data.sqlite"
sdm_path = TARGET_DB_PATH

# Open both databases
source_conn = sqlite3.connect(crm_path)
source_conn.row_factory = sqlite3.Row
source_cursor = source_conn.cursor()

sdm_conn = sqlite3.connect(sdm_path)
sdm_cursor = sdm_conn.cursor()

# Try to get first row from crm_country
source_cursor.execute("SELECT * FROM crm_country LIMIT 1")
row = source_cursor.fetchone()

if row:
    row_dict = dict(row)
    print(f"Row data: {row_dict}")
    
    cols = ", ".join(row_dict.keys())
    placeholders = ", ".join(["?" for _ in row_dict])
    sql = f"INSERT INTO CRM_crm_country ({cols}) VALUES ({placeholders})"
    
    print(f"SQL: {sql}")
    print(f"Values: {tuple(row_dict.values())}")
    
    try:
        sdm_cursor.execute(sql, tuple(row_dict.values()))
        sdm_conn.commit()
        print("✓ INSERT successful!")
    except Exception as e:
        print(f"✗ ERROR: {type(e).__name__}: {e}")

source_conn.close()
sdm_conn.close()



Testing crm_country load...
Row data: {'COUNTRY_CODE': '1', 'COUNTRY_EN': 'France', 'FLAG_IMAGE': 'F01', 'SALES_TERRITORY_CODE': '6'}
SQL: INSERT INTO CRM_crm_country (COUNTRY_CODE, COUNTRY_EN, FLAG_IMAGE, SALES_TERRITORY_CODE) VALUES (?, ?, ?, ?)
Values: ('1', 'France', 'F01', '6')
✗ ERROR: IntegrityError: UNIQUE constraint failed: CRM_crm_country.COUNTRY_CODE


In [ ]:
# Check if tables exist in SDM

print("Checking table existence in SDM...")
print("=" * 60)

sdm_conn = sqlite3.connect(TARGET_DB_PATH)
sdm_cursor = sdm_conn.cursor()

tables_to_check = [
    'CRM_crm_country',
    'CRM_customer',
    'CRM_customer_headquarters',
    'SALES_order_details',
    'SALES_order_header',
    'STAFF_sales_representative',
]

for table_name in tables_to_check:
    try:
        sdm_cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = sdm_cursor.fetchone()[0]
        
        # Try PRAGMA
        sdm_cursor.execute(f"PRAGMA table_info({table_name})")
        cols = sdm_cursor.fetchall()
        
        print(f"✓ {table_name:40} exists with {len(cols)} columns, {count} rows")
    except Exception as e:
        print(f"✗ {table_name:40} ERROR: {e}")

sdm_conn.close()


Checking table existence in SDM...
✓ CRM_crm_country                          exists with 4 columns, 19 rows
✓ CRM_customer                             exists with 4 columns, 109 rows
✓ CRM_customer_headquarters                exists with 11 columns, 414 rows
✓ SALES_order_details                      exists with 7 columns, 40990 rows
✓ SALES_order_header                       exists with 8 columns, 4968 rows
✓ STAFF_sales_representative               exists with 11 columns, 102 rows


In [ ]:
# Mimic exact load_sqlite_table_incremental steps for crm_country

print("\nMIMICKING load_sqlite_table_incremental for crm_country")
print("=" * 80)

crm_path = GO_DIR / "CRM-data.sqlite"
sdm_path = TARGET_DB_PATH

# Step 1: Get schema
print("Step 1: Get schema for crm_country...")
try:
    schema = get_sqlite_schema(crm_path)
    table_schema = schema.get('crm_country', {})
    pk_columns = table_schema.get('primary_key', [])
    print(f"  ✓ PK columns: {pk_columns}")
except Exception as e:
    print(f"  ✗ ERROR: {e}")
    
# Step 2: Get row hashes from metadata
print("\nStep 2: Get row hashes from metadata...")
try:
    sdm_conn = sqlite3.connect(sdm_path)
    stored_hashes = get_row_hashes(sdm_conn, "CRM_crm_country")
    print(f"  ✓ Stored hashes: {len(stored_hashes)} entries")
except Exception as e:
    print(f"  ✗ ERROR: {e}")

# Step 3: Get source data
print("\nStep 3: Get source data...")
try:
    source_conn = sqlite3.connect(crm_path)
    source_conn.row_factory = sqlite3.Row
    source_cursor = source_conn.cursor()
    
    source_cursor.execute("SELECT * FROM crm_country")
    rows = source_cursor.fetchall()
    print(f"  ✓ Got {len(rows)} rows from source")
except Exception as e:
    print(f"  ✗ ERROR: {e}")

# Step 4: Try first 3 inserts
print("\nStep 4: Try loading first 3 rows...")
try:
    sdm_cursor = sdm_conn.cursor()
    
    for idx, row in enumerate(rows[:3]):
        row_dict = dict(row)
        
        cols = ", ".join(row_dict.keys())
        placeholders = ", ".join(["?" for _ in row_dict])
        insert_sql = f"INSERT INTO CRM_crm_country ({cols}) VALUES ({placeholders})"
        
        try:
            sdm_cursor.execute(insert_sql, tuple(row_dict.values()))
            print(f"  ✓ Row {idx}: INSERT successful")
        except Exception as e:
            print(f"  ✗ Row {idx}: {type(e).__name__}: {e}")
    
    sdm_conn.commit()
except Exception as e:
    print(f"  ✗ ERROR: {e}")

# Verify
print("\nVerify: Count rows in CRM_crm_country...")
try:
    sdm_cursor.execute("SELECT COUNT(*) FROM CRM_crm_country")
    count = sdm_cursor.fetchone()[0]
    print(f"  ✓ CRM_crm_country now has {count} rows")
except Exception as e:
    print(f"  ✗ ERROR: {e}")

source_conn.close()
sdm_conn.close()

print("\n" + "=" * 80)



MIMICKING load_sqlite_table_incremental for crm_country
Step 1: Get schema for crm_country...
  ✓ PK columns: ['COUNTRY_CODE']

Step 2: Get row hashes from metadata...
  ✓ Stored hashes: 19 entries

Step 3: Get source data...
  ✓ Got 19 rows from source

Step 4: Try loading first 3 rows...
  ✗ Row 0: IntegrityError: UNIQUE constraint failed: CRM_crm_country.COUNTRY_CODE
  ✗ Row 1: IntegrityError: UNIQUE constraint failed: CRM_crm_country.COUNTRY_CODE
  ✗ Row 2: IntegrityError: UNIQUE constraint failed: CRM_crm_country.COUNTRY_CODE

Verify: Count rows in CRM_crm_country...
  ✓ CRM_crm_country now has 19 rows



In [ ]:
# Final Summary: SDM Load Complete

import sqlite3

conn = sqlite3.connect(TARGET_DB_PATH)
cursor = conn.cursor()

cursor.execute("""
    SELECT name FROM sqlite_master 
    WHERE type='table' AND name NOT LIKE 'sqlite_%' AND name != 'sdm_load_metadata'
    ORDER BY name
""")
tables = cursor.fetchall()

empty_count = 0
total_rows = 0
loaded_count = 0

print("\n" + "=" * 80)
print("✅ SDM DATABASE LOAD COMPLETE")
print("=" * 80 + "\n")

for (table_name,) in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    count = cursor.fetchone()[0]
    total_rows += count
    
    if count == 0:
        empty_count += 1
    else:
        loaded_count += 1

print(f"SUMMARY:")
print(f"  Total Tables:    {len(tables)}")
print(f"  Tables Loaded:   {loaded_count}")
print(f"  Empty Tables:    {empty_count}")
print(f"  Total Rows:      {total_rows:,}")

if empty_count == 0:
    print(f"\n✨ SUCCESS! All {len(tables)} tables have been populated with data!\n")
else:
    print(f"\n⚠️  {empty_count} tables still empty\n")

# Show database file size
db_size = TARGET_DB_PATH.stat().st_size / (1024 * 1024)
print(f"Database File Size: {db_size:.2f} MB")

print("\n" + "=" * 80 + "\n")

conn.close()



✅ SDM DATABASE LOAD COMPLETE

SUMMARY:
  Total Tables:    31
  Tables Loaded:   31
  Empty Tables:    0
  Total Rows:      99,277

✨ SUCCESS! All 31 tables have been populated with data!

Database File Size: 12.81 MB


